# Classifying Names: Feed-Forward Network vs. RNN

In this notebook we classify surnames by their language of origin (18 languages, e.g. *Russian*, *German*, *Japanese*).
A name is a **sequence of characters**, so it is a small but complete NLP problem.

We solve it twice:

* **Part A**: with a normal feed-forward network.
  For this we have to force every name into a fixed-length vector.
* **Part B**: with a character-level RNN that you implement yourself.
  It reads the name character by character and carries a hidden state from step to step.

The `assert` cells check your solution. If an assert fails, read the message and fix your code before you continue.

## Preparing the data

The folder `data/names` contains 18 files `[Language].txt` with one name per line.
The names are already cleaned: accents are removed (`Ślusàrski` → `Slusarski`) and only the 52 ASCII letters are kept.
The label `y` is the index of the language in `all_categories`.

In [1]:
import glob, os, string, time
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

torch.manual_seed(0)

filenames = sorted(glob.glob('../data/names/*.txt'))   # sorted: same label order on every OS
all_categories = [os.path.splitext(os.path.basename(f))[0] for f in filenames]
n_categories = len(all_categories)

all_letters = string.ascii_letters                   # our vocabulary: a-z, A-Z
n_letters = len(all_letters)

X, y = [], []
for index, filename in enumerate(filenames):
    names = open(filename).read().split()
    X += names
    y += [index] * len(names)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

print(n_categories, "languages,", n_letters, "letters,", len(X_train), "train /", len(X_test), "test names")
print("Examples:", [(X_train[i], all_categories[y_train[i]]) for i in range(5)])

18 languages, 52 letters, 14394 train / 3599 test names
Examples: [('Egerton', 'English'), ('Bakaloff', 'Russian'), ('Avtomovitch', 'Russian'), ('Guzun', 'Russian'), ('Hismatullin', 'Russian')]


Let's look at how the names are distributed over the languages:

In [2]:
from collections import Counter

counts = Counter(y_train)
for index, count in counts.most_common():
    print(f"{all_categories[index]:12s} {count:5d}  ({count / len(y_train):.1%})")

Russian       7466  (51.9%)
English       2951  (20.5%)
Japanese       766  (5.3%)
German         563  (3.9%)
Italian        559  (3.9%)
Czech          395  (2.7%)
Spanish        238  (1.7%)
Dutch          231  (1.6%)
French         220  (1.5%)
Chinese        196  (1.4%)
Irish          192  (1.3%)
Greek          150  (1.0%)
Polish         113  (0.8%)
Arabic          89  (0.6%)
Scottish        78  (0.5%)
Korean          75  (0.5%)
Vietnamese      56  (0.4%)
Portuguese      56  (0.4%)


### Task 0: Majority baseline

The dataset is very imbalanced. Before we train anything, we need a reference point:
what test accuracy does a "model" reach that **always predicts the most frequent class of the training data**?
Store the result in `majority_acc`. Every model below has to beat this number clearly.

In [3]:
majority_class = counts.most_common(1)[0][0]
majority_acc = sum(1 for label in y_test if label == majority_class) / len(y_test)
print(f"Majority baseline ({all_categories[majority_class]}): {majority_acc:.3f}")

Majority baseline (Russian): 0.521


In [4]:
assert majority_acc is not None, "Compute majority_acc first."
assert 0.50 < majority_acc < 0.54, f"Expected about 0.52, got {majority_acc}"
print("OK")

OK


---
## Part A: A feed-forward network for names

A feed-forward network needs an input vector of **fixed size**. Names have different lengths (2 to 19 characters).
The usual trick: choose a maximum length `max_len`, cut longer names and fill shorter ones with zeros (*padding*).
Each position gets a one-hot vector of length 52, and all positions are concatenated into one long vector:

```
"Abe", max_len=4:   [ one-hot('A') | one-hot('b') | one-hot('e') | 0 0 ... 0 ]   -> 4 * 52 = 208 values
```

### Task 1: Fixed-length encoding

Write `nameToFixedTensor(name, max_len)` that returns a 1-D float tensor of size `max_len * n_letters`.
Characters after position `max_len` are dropped.

In [15]:
def nameToFixedTensor(name, max_len=12):
    tensor = torch.zeros(max_len, n_letters)
    for position, letter in enumerate(name[:max_len]):
        tensor[position][all_letters.find(letter)] = 1
    return tensor.flatten()

In [18]:
t = nameToFixedTensor("Abe", max_len=12)
#print(t)
assert t is not None and t.shape == (12 * n_letters,), "Expected a 1-D tensor of size max_len * n_letters"
assert t.sum() == 3, "Exactly one 1 per character"
assert t.view(12, n_letters)[0, all_letters.find('A')] == 1, "'A' must be at position 0"
assert t.view(12, n_letters)[2, all_letters.find('e')] == 1, "'e' must be at position 2"
assert nameToFixedTensor("Abcdefghijklmnop", max_len=12).sum() == 12, "Names longer than max_len are cut"
print("OK")

OK


### Task 2: Train a feed-forward network

1. Encode all training and test names with `nameToFixedTensor` (use `max_len = 12`) and stack them into
   two matrices `X_train_fixed` and `X_test_fixed`. Convert the labels to tensors.
2. Define a class `MLP` with one hidden layer of 128 neurons (ReLU) and one output neuron per language.
   Define the layers as named attributes in `__init__` and write `forward` yourself.
3. Train it with mini-batches (batch size 64), `CrossEntropyLoss` and Adam (`lr=1e-3`) for 20 epochs.
   This takes well under a minute.
4. Compute the test accuracy and store it in `mlp_acc`.

In [7]:
max_len = 12
X_train_fixed = torch.stack([nameToFixedTensor(name, max_len) for name in X_train])
X_test_fixed = torch.stack([nameToFixedTensor(name, max_len) for name in X_test])
y_train_t = torch.tensor(y_train)
y_test_t = torch.tensor(y_test)

class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

torch.manual_seed(0)
mlp = MLP(max_len * n_letters, 128, n_categories)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3)

for epoch in range(20):
    permutation = torch.randperm(len(X_train_fixed))
    for i in range(0, len(permutation), 64):
        batch = permutation[i:i + 64]
        optimizer.zero_grad()
        loss = criterion(mlp(X_train_fixed[batch]), y_train_t[batch])
        loss.backward()
        optimizer.step()

with torch.no_grad():
    mlp_acc = (mlp(X_test_fixed).argmax(dim=1) == y_test_t).float().mean().item()
print(f"MLP test accuracy: {mlp_acc:.3f}")

MLP test accuracy: 0.780


In [8]:
assert mlp_acc is not None, "Compute mlp_acc first."
assert mlp_acc > 0.72, f"Expected more than 0.72, got {mlp_acc:.3f}. Check encoding, learning rate and number of epochs."
print(f"OK: MLP {mlp_acc:.3f} vs. majority baseline {majority_acc:.3f}")

OK: MLP 0.780 vs. majority baseline 0.521


### Task 3: Where are the limits?

The MLP works surprisingly well on names. Still, the approach has structural problems.
Answer the questions (in a markdown cell below or in your notes). Short code experiments are welcome.

1. What happens with the name `"Andriyashevskiy"` (15 letters) when `max_len = 12`? Which part of the name gets lost,
   and why is this part particularly informative for many languages?
2. How many weights does the first layer have for `max_len = 12`? How many would it have for whole sentences
   with `max_len = 50` *words* and a vocabulary of 10,000 words?
3. The suffix `-ski` appears at position 4–6 in `"Kowalski"` and at position 6–8 in `"Wisniewski"`.
   Which weights of the first layer are responsible for recognizing it in each case? What does this mean for learning?

**Answers:**

1. The input only keeps `"Andriyashevs"`, the ending `-kiy` is cut. Endings (`-ov`, `-ski`, `-kiy`, `-ez`, `-escu`, ...) are often
   the most informative part of a surname, so the truncation removes exactly the signal we need. Increasing `max_len` helps,
   but there is always a longer input, and most inputs then consist mostly of padding.
2. Names: $12 \cdot 52 \cdot 128 = 79{,}872$ weights (+128 biases). Sentences: $50 \cdot 10{,}000 \cdot 128 = 64{,}000{,}000$ weights
   in the first layer alone. The number of parameters grows linearly with the maximum length.
3. Different weights: each position has its own block of weights in `fc1`. What the network learns about `-ski` at positions
   4–6 does not help at positions 6–8. The same pattern has to be learned again for every position, which needs more data.
   An RNN uses **the same weights at every time step** (weight sharing over time), just like a CNN uses the same filter at every
   location of an image.